# 03 · Load the five customer sources into Turso
Creates five source tables in Turso (`crm_accounts`, `billing_customers`, `support_contacts`, `web_visitors`, `marketing_leads`) and loads one CSV batch of each (upsert, so re-applying a batch is safe).

* **batch 1**: initial load, overlapping on email, with conflicting values, duplicates and bad rows
* **batch 2**: plan changes, cancellations, new support contacts, more visits, and four leads that convert into CRM accounts and paying customers

Needs `00_config.ipynb` (already in your project). Do **not** re-run batch 1 after batch 2 unless you reset first.

In [1]:
import os
os.chdir("/workspaces/pythonProject/medallion_v2/notebooks")

In [2]:
import os
BATCH = int(os.getenv("BATCH", 1))              # 1 or 2: change the number, then run
RESET = os.getenv("RESET_SOURCE") == "1"        # True drops the five tables first (start over)

In [3]:
%run -i ./00_config.ipynb

cwd : /workspaces/pythonProject/medallion_v2/notebooks
ROOT: /workspaces/pythonProject/medallion_v2
LAKE: /workspaces/pythonProject/medallion_v2/lake
rules folder exists: True | file there: True
rule library: 22 checks for 8 entities


In [4]:
def find(name):
    for d in (Path.cwd(), ROOT, ROOT / "data", ROOT / "sql"):
        if (d / name).exists():
            return d / name
    raise FileNotFoundError(f"{name} not found next to the notebook, in data/ or in sql/")

TABLES = {"crm_accounts": "account_id", "billing_customers": "billing_id", "support_contacts": "contact_id",
          "web_visitors": "visitor_id", "marketing_leads": "lead_id"}

conn = source_conn()
print("target:", "Turso" if TURSO_URL else "local SQLite stand-in")
if RESET:
    for t in TABLES: conn.execute(f"DROP TABLE IF EXISTS {t}")

ddl = "\n".join(l for l in find("schema_multisource.sql").read_text().splitlines() if not l.strip().startswith("--"))
for stmt in [s.strip() for s in ddl.split(";") if s.strip()]:
    conn.execute(stmt)
conn.commit()

target: Turso


In [5]:
def load_csv(table, path):
    df = pd.read_csv(path, dtype=str, keep_default_na=False)
    cols, key = list(df.columns), TABLES[table]
    rows = [tuple(None if v == "" else v for v in r) for r in df.itertuples(index=False)]   # blank -> NULL
    upd = ", ".join(f"{c} = excluded.{c}" for c in cols if c != key)
    sql = (f"INSERT INTO {table} ({', '.join(cols)}) VALUES ({', '.join('?' * len(cols))}) "
           f"ON CONFLICT({key}) DO UPDATE SET {upd}")
    for r in rows: conn.execute(sql, r)
    conn.commit()
    return len(rows)

for table in TABLES:
    try:
        f = find(f"{table}_batch{BATCH}.csv")
    except FileNotFoundError:
        continue
    print(f"{table:18s} loaded {load_csv(table, f):4d} rows from {f.name}")
try: conn.sync()
except Exception: pass

for t in TABLES:
    cur = conn.cursor(); cur.execute(f"SELECT count(*), max(updated_at) FROM {t}"); print(t, cur.fetchone())

crm_accounts       loaded   60 rows from crm_accounts_batch1.csv
billing_customers  loaded   47 rows from billing_customers_batch1.csv
support_contacts   loaded   34 rows from support_contacts_batch1.csv
web_visitors       loaded   31 rows from web_visitors_batch1.csv
marketing_leads    loaded   35 rows from marketing_leads_batch1.csv
crm_accounts (60, '2026-10-30 14:13:55')
billing_customers (47, '2026-10-31 20:29:53')
support_contacts (34, '2026-10-31 21:16:00')
web_visitors (31, '2026-10-31 18:55:13')
marketing_leads (35, '2026-10-28 20:20:37')
